# Instance-Aware Depth on Colab

Runs the project's code on a Colab GPU. **Runtime → Change runtime type → T4 GPU** first.

**Data** (first match wins):
1. `MyDrive/instance-aware-depth/iad_data.tar` (your own copy), or
2. `MyDrive/iad_data.tar`: a **shortcut to the team's shared, view-only bundle**. In Drive, open
   "Shared with me", right-click `iad_data.tar` → Organize → Add shortcut → My Drive.
3. Neither: the notebook downloads and prepares everything (~10 min) and saves your own copy.

Checkpoints and results are always saved to **your own** `MyDrive/instance-aware-depth/`, never to the shared file.

**Private repo:** add a GitHub token as a Colab secret named `GH_TOKEN` (🔑 icon on the left).
Not needed once the repo is public.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 1. Settings and Google Drive

In [ ]:
import os
REPO = 'csgomez25/instance-aware-depth'
BRANCH = 'main'
DRIVE_DIR = '/content/drive/MyDrive/instance-aware-depth'   # YOUR folder: checkpoints, results, own data copy

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['DATA_ROOT'] = '/content/data'                     # local disk: much faster than reading from Drive
os.environ['CKPT_DIR'] = f'{DRIVE_DIR}/checkpoints'
os.environ['RUNS_DIR'] = f'{DRIVE_DIR}/runs'                # training runs survive disconnects

## 2. Get the code

In [ ]:
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.isdir('/content/iad'):
    !git clone -q -b {BRANCH} {url} /content/iad
else:
    !git -C /content/iad pull -q
%cd /content/iad
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y rsync > /dev/null

## 3. Data: restore the bundle from Drive, or build it once

In [ ]:
own_bundle = f'{DRIVE_DIR}/iad_data.tar'
shared_bundle = '/content/drive/MyDrive/iad_data.tar'          # shortcut to the team's view-only copy
bundle = next((b for b in (own_bundle, shared_bundle) if os.path.exists(b)), None)
os.makedirs(os.environ['DATA_ROOT'], exist_ok=True)
if bundle:
    print('restoring data from', bundle)
    data_root = os.environ['DATA_ROOT']
    !tar -xf "{bundle}" -C "{data_root}"                       # read-only: the bundle is never modified
!bash scripts/setup.sh      # fresh: downloads + prepares everything; restored: only code + checkpoints
if not bundle:
    !bash scripts/pack_data.sh "{own_bundle}"                  # saved to your own Drive folder only
!ls $DATA_ROOT $DATA_ROOT/nyu

## 4. Smoke test (~1 min)

In [ ]:
%cd /content/iad/tools
!python infer.py $DATA_ROOT/nyu/rgb/000500.png --outdir /content/iad/results/infer
!python eval_nyu.py --split val --limit 10 --tag colab_smoke

## 5. Baseline evaluations (optional, ~5 min)
Use `--split val` while tuning; `--split test` only for final numbers.

In [ ]:
!python eval_nyu.py --split val  --gt filled
!python eval_nyu.py --split test --gt filled
!python eval_nyu.py --split test --gt raw
!python eval_da2k.py

## 6. Training
Runs are saved to `MyDrive/instance-aware-depth/runs/<name>/` (`log.csv`, `best.pth`).
If Colab disconnects, reconnect, run sections 1–3 again, then **re-run the same training cell**: it resumes
from the last finished epoch. A finished run just prints its result. Use a new `--name` for new settings.

Each run uses the val split for model selection; the test split is not touched.

In [ ]:
%cd /content/iad/tools
# Quick check that training works on this GPU (~1 min); prints speed and memory per epoch.
!python train_nyu.py --name colab_smoke --epochs 1 --max-steps 20 --bs 8

### H1: baseline fine-tune, filled vs raw depth target (decides the target for every later run)

In [ ]:
!python train_nyu.py --name h1_silog_filled --train-gt filled --bs 4 --epochs 20

In [ ]:
!python train_nyu.py --name h1_silog_raw --train-gt raw --bs 4 --epochs 20

### H2: instance-aware loss ablations (after H1; set TARGET to the H1 winner)
Each line is one run; re-running the cell skips finished runs and resumes an interrupted one.

In [ ]:
TARGET = 'filled'   # H1 winner (filled 0.0910 vs raw 0.0944, mean of last 5 val epochs)
M = 'yolo11s-seg'
# lambda = 0 control: loading masks changes the random crops, so this, not h1_silog_filled, is the paired baseline
!python train_nyu.py --name h2_silog_yolo_l0     --train-gt {TARGET} --masks {M}
!python train_nyu.py --name h2_edge_yolo_l0.5    --train-gt {TARGET} --masks {M}  --lambda-edge 0.5
!python train_nyu.py --name h2_obj_yolo_l0.5     --train-gt {TARGET} --masks {M}  --lambda-obj 0.5
!python train_nyu.py --name h2_both_yolo_l0.5    --train-gt {TARGET} --masks {M}  --lambda-edge 0.5 --lambda-obj 0.5
!python train_nyu.py --name h2_both_gt_l0.5      --train-gt {TARGET} --masks gt   --lambda-edge 0.5 --lambda-obj 0.5

### H2b: edge-loss weight sweep
The edge term carries the boundary gain and the object term adds nothing on edges, so sweep `lambda_edge` alone.

In [ ]:
%cd /content/iad/tools
for lam in ['0.25', '1.0', '2.0']:
    !python train_nyu.py --name h2_edge_yolo_l{lam} --train-gt {TARGET} --masks {M} --lambda-edge {lam}

### H5: seeds of the control and the chosen method (lambda_edge = 1)
Decided after H2b. Keep all six runs on one machine; the laptop queue (`runs/h5_queue.sh`) is the alternative.

In [ ]:
%cd /content/iad/tools
for seed in [0, 1, 2]:
    for lam in ['0', '1.0']:
        !python train_nyu.py --name h5_edge_yolo_l{lam}_s{seed} --train-gt filled --masks yolo11s-seg --lambda-edge {lam} --seed {seed}

### H3: learning rate x batch size sweep on the SiLog baseline (one machine for the whole sweep)
**Already done on the laptop (results in `results/h3_val.csv`). Skip this cell** unless the sweep is being redone.

In [ ]:
TARGET = 'filled'   # <- H1 winner
for lr in ['2e-6', '5e-6', '1e-5']:
    for bs in [2, 4, 8]:
        !python train_nyu.py --name h3_silog_lr{lr}_bs{bs} --train-gt {TARGET} --lr {lr} --bs {bs}

In [ ]:
# Compare runs: best epoch (optimistic: lowest of many noisy scores) and mean of the last 5 epochs (steadier)
import glob, json, pandas as pd
rows = []
for f in sorted(glob.glob(f"{os.environ['RUNS_DIR']}/*/log.csv")):
    log = pd.read_csv(f); name = f.split('/')[-2]
    b = log.loc[log.val_filled_abs_rel.idxmin()]
    last5 = log.tail(5)
    rows.append({'run': name, 'epochs_done': len(log), 'best_epoch': int(b.epoch),
                 'best_val_abs_rel': b.val_filled_abs_rel, 'best_val_abs_rel_raw': b.val_raw_abs_rel, 'best_val_d1': b.val_filled_d1,
                 'last5_val_abs_rel': round(last5.val_filled_abs_rel.mean(), 4),   # steadier than the single best epoch
                 'last5_val_d1': round(last5.val_filled_d1.mean(), 4)})
pd.DataFrame(rows).sort_values('last5_val_abs_rel')

### Boundary metrics for finished runs
Scores each run's `best.pth` on val: AbsRel etc. plus `bf1`, `bf1_obj` (object boundaries) and `flying`
(see `tools/boundary.py`). About 20 s per run on a T4. Section 8 saves the CSVs to Drive.

In [ ]:
%cd /content/iad/tools
RD = os.environ['RUNS_DIR']   # build paths in Python: '$RUNS_DIR/{run}' in a ! line leaves {run} unexpanded
RUNS = ['h1_silog_filled', 'h2_silog_yolo_l0', 'h2_edge_yolo_l0.5', 'h2_obj_yolo_l0.5', 'h2_both_yolo_l0.5', 'h2_both_gt_l0.5',
        'h2_edge_yolo_l0.25', 'h2_edge_yolo_l1.0', 'h2_edge_yolo_l2.0']
for run in RUNS:
    ckpt = f"{RD}/{run}/best.pth"
    if os.path.exists(ckpt):
        !python eval_nyu.py --split val --gt filled --ckpt "{ckpt}" --tag {run} 2>&1 | grep -v xFormers | tail -2
    else:
        print('not trained yet:', run)

### iBims-1 zero-shot (second dataset)
100 indoor images the model never trains on: depth metrics in metres and scale-aligned (`sa_*`), official iBims
boundary (`dbe_*`), directed-depth (`dde_*`) and planarity (`pe_*`) errors, plus `bf1` / `flying`. About 1 min per model.

In [ ]:
%cd /content/iad/tools
!python eval_ibims.py --tag pretrained_hypersim_vits 2>&1 | grep -v xFormers | tail -2
for run in ['h1_silog_filled', 'h2_silog_yolo_l0']:
    ckpt = f"{RD}/{run}/best.pth"
    if os.path.exists(ckpt):
        !python eval_ibims.py --ckpt "{ckpt}" --tag {run} 2>&1 | grep -v xFormers | tail -2

## 7. Final test numbers (only once experiments are frozen)

In [ ]:
RUN = 'h5_edge_yolo_l1.0_s0'   # the frozen method; repeat for the control and the zero-shot model
ckpt = f"{os.environ['RUNS_DIR']}/{RUN}/best.pth"
!python eval_nyu.py --split test --gt filled --ckpt "{ckpt}" --tag {RUN}
!python eval_nyu.py --split test --gt raw    --ckpt "{ckpt}" --tag {RUN}
!python eval_ibims.py --ckpt "{ckpt}" --tag {RUN}

## 8. Save results to Drive

In [ ]:
import shutil, time
dst = f'{DRIVE_DIR}/results/{time.strftime("%Y%m%d-%H%M%S")}'
shutil.copytree('/content/iad/results', dst, ignore=shutil.ignore_patterns('*.ply', '*.npy'))
print('saved to', dst)